In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# HyDE

## 假设文档生成

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """您是一套名为 LangChain、LangGraph、LangServe 和 LangSmith 的软件的专家，这套软件用于构建由 LLM 驱动的应用程序。

LangChain 是一个 Python 框架，它提供了大量集成，可以很容易地组合起来构建 LLM 应用程序。
LangGraph 是构建在 LangChain 基础上的 Python 软件包，可轻松构建有状态、多角色的 LLM 应用程序。
LangServe 是一个构建在 LangChain 基础上的 Python 软件包，可以轻松地将 LangChain 应用程序部署为 REST API。
LangSmith 是一个可轻松跟踪和测试 LLM 应用程序的平台。

尽可能回答用户的问题。就像编写教程一样回答用户的问题。"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(temperature=0)
qa_no_context = prompt | llm | StrOutputParser()

In [3]:
answer = qa_no_context.invoke(
    {
        "question": "如何在链中使用多模型，并将链转换为 REST API？"
    }
)
print(answer)

在 LangChain 中使用多模型并将链转换为 REST API 是非常简单的。以下是一些基本步骤：

1. **使用多模型**：
   - 在 LangChain 中使用多模型可以通过 LangGraph 实现。LangGraph 允许您轻松地构建有状态、多角色的 LLM 应用程序。
   - 首先，您需要定义每个模型及其对应的角色。然后，您可以使用 LangGraph 中的功能将这些模型组合在一起，形成一个有状态的链。
   - 您可以定义不同的角色来处理不同的任务，每个角色可以使用不同的模型来执行特定的功能。

2. **将链转换为 REST API**：
   - 一旦您在 LangChain 中构建了多模型的链，您可以使用 LangServe 将其部署为 REST API。
   - LangServe 提供了简单易用的功能，可以将您的 LangChain 应用程序快速转换为可以通过 HTTP 请求访问的 API。
   - 您可以定义端点和请求处理程序，以便通过 API 调用执行链中定义的功能。

下面是一个简单的示例代码，演示如何在 LangChain 中使用多模型并将其转换为 REST API：

```python
from langchain import LangGraph, LangServe

# 定义多个模型
model1 = YourModel1()
model2 = YourModel2()

# 定义角色和功能
roles = {
    'role1': model1,
    'role2': model2
}

# 创建 LangGraph 实例
graph = LangGraph(roles)

# 创建 LangServe 实例
serve = LangServe(graph)

# 将链转换为 REST API
serve.serve()
```

通过上述步骤，您可以在 LangChain 中使用多模型，并将其转换为 REST API，以便通过 HTTP 请求访问您的应用程序。希望这可以帮助您开始构建您的应用程序！如果您有任何进一步的问题，请随时问我。


### 返回假设文档和原始问题

In [4]:
from langchain_core.runnables import RunnablePassthrough

hyde_chain = RunnablePassthrough.assign(hypothetical_document=qa_no_context)

hyde_chain.invoke(
    {
        "question": "如何在链中使用多模型，并将链转换为 REST API？"
    }
)

{'question': '如何在链中使用多模型，并将链转换为 REST API？',
 'hypothetical_document': '在 LangChain 中使用多模型并将链转换为 REST API 是非常简单的。以下是一些步骤来实现这一目标：\n\n1. **使用多模型**：\n   - 在 LangChain 中使用多模型可以通过 LangGraph 来实现。LangGraph 允许您构建有状态、多角色的 LLM 应用程序，其中可以包含多个模型。\n   - 首先，您需要定义每个模型及其角色。然后，您可以使用 LangGraph 中的功能将这些模型组合在一起，形成一个有状态的链。\n   - 您可以定义每个模型的输入、输出以及模型之间的交互方式，以实现复杂的应用程序逻辑。\n\n2. **将链转换为 REST API**：\n   - 一旦您在 LangChain 中构建了包含多个模型的链，您可以使用 LangServe 将这个链转换为 REST API。\n   - LangServe 提供了简单易用的功能，可以将您的 LangChain 应用程序快速部署为 REST API，使其可以通过 HTTP 请求进行访问。\n   - 您可以定义每个模型的输入和输出格式，并将其映射到 REST API 的端点上。这样，其他应用程序或服务就可以通过 API 调用您的链中的模型了。\n\n综上所述，您可以通过 LangGraph 在 LangChain 中使用多模型，并通过 LangServe 将整个链转换为 REST API。这样，您可以构建复杂的 LLM 应用程序，并通过 API 提供给其他应用程序使用。'}

# 优化Naive-RAG

## 索引

In [5]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

## 检索

In [6]:
from langchain.prompts import ChatPromptTemplate

# HyDE document genration
template = """请撰写一篇科学论文段落来回答以下问题：
问题: {question}
段落："""

prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Chain
generate_docs_for_retrieval = (
    prompt_hyde | ChatOpenAI(temperature=0) | StrOutputParser() 
)

In [7]:
question = "什么是few shot？"

In [8]:
generate_docs_for_retrieval.invoke({"question": question})

'Few shot learning是一种机器学习方法，其目标是在只有少量标记数据可用的情况下进行学习。在传统的监督学习中，通常需要大量的标记数据来训练模型，以便模型能够准确地泛化到新的未见数据。然而，在现实世界中，获取大量标记数据可能是困难或昂贵的。因此，few shot learning旨在解决这一问题，通过利用少量标记数据来训练模型，使其能够在面对新的未见数据时表现良好。Few shot learning的研究对于推动机器学习领域的发展具有重要意义，可以帮助我们更好地利用有限的数据资源来构建高效的模型。'

In [9]:
# Retrieve
new_retriever = generate_docs_for_retrieval | retriever
# new_retriever = prompt_hyde | ChatOpenAI(temperature=0) | StrOutputParser() | retriever
retireved_docs = new_retriever.invoke({"question": question})
retireved_docs

[Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'})

## 生成

In [10]:
# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": new_retriever | format_docs, "question": RunnablePassthrough()}
    # {"context": prompt_hyde | llm | output_parser | retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# Run
question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)

Q: 什么是few shot？
A: 
少样本学习（Few-shot learning）是一种模型提示方法，通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而提高性能。这种方法通常比零样本学习表现更好，但可能消耗更多Token并受到上下文长度限制的影响。
